# Controlled Decoding: шаг 4, проверки по фидбеку руководителя (09.11)

| Пункт фидбека | Раздел | GPU | Время (полный прогон, 2×T4) |
|---|---|---|---|
| 4. Длина ответа, доля законченных, reward только на законченных | 1 | нет | секунды |
| 5. pair_acc@t только на парах, где оба ответа длиннее t | 2 | scorer | ~5 мин |
| 3. Другой судья: две независимые RM | 3 | судьи | ~1–1,5 ч |
| 2. Реальная стоимость: время генерации и вызовов scorer'а на 50 промптах | 4 | всё | ~20 мин |
| 6. Температура: best-of-K и 3 конфигурации CD при T=0.7, top-p=0.9 с тем же scorer'ом | 5 | всё | ~2 ч |

**Входы (Add Input):** `cd-hh-rollouts`, `cd-hh-scorer` и шарды полного прогона `03` (датасет `cd-hh-eval`
или Output версии ноутбука `03`: нужна папка с `run_meta.json`, `bon_*.parquet`, `cd_M*_K*_*.parquet`).

**Промежуточные сохранения:** всё, что требует GPU, считается шардами по 50 промптов и сохраняется в `cd_feedback`.
Restart и Run All пропускают готовое. Для новой сессии подключи датасет, который загружает последняя ячейка, и готовые
шарды подтянутся из `/kaggle/input`. Отдельные разделы можно выключить в `RUN`.

In [ ]:
import os, re, json, glob, time, shutil, math
from collections import defaultdict
from contextlib import contextmanager
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForTokenClassification, AutoModelForSequenceClassification

print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
DEBUG = True  # 16 промптов и 8 для замеров времени; для полного прогона поставить False

RUN = {"pair_acc": True, "judges": True, "timing": True, "temperature": True}  # разделы с GPU

N_PROMPTS = 16 if DEBUG else 500   # первые N test-промптов, те же, что в 03
PAIR_TS = [8, 16, 32, 64, 128]
PAIR_MIN_GAP = 0.5                 # как в 02: пары с почти равным reward не считаем
JUDGES = {                         # независимые судьи; цель scorer'а была Skywork-Reward-V2-Qwen3-0.6B
    "grm_llama3b": "Ray2333/GRM-Llama3.2-3B-rewardmodel-ft",  # другая команда и базовая модель, RewardBench 90.9
    "skywork_qwen4b": "Skywork/Skywork-Reward-V2-Qwen3-4B",   # та же серия, но в 7 раз больше
}
N_TIMING = 8 if DEBUG else 50
TIMING_BON_KS = [2, 4, 8, 16, 32]
TEMP_KW = {"temperature": 0.7, "top_p": 0.9}
TEMP_CD = [(16, 4), (32, 4), (16, 8)]  # (M, K)
TEMP_BON_K = 32

SHARD_SIZE = 8 if DEBUG else 50
GEN_SEQS = 128
SCORER_BATCH = 32
RM_BATCH = 32
JUDGE_BATCH = 16
N_BOOT = 5000
SEED = 1

GEN_DEVICE = "cuda:0"
AUX_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
DTYPE = torch.float16

WORK = "/kaggle/working" if os.path.exists("/kaggle") else "."
OUT = f"{WORK}/cd_feedback_debug" if DEBUG else f"{WORK}/cd_feedback"
os.makedirs(OUT, exist_ok=True)

## Входы

In [ ]:
def find_one(pattern, what):
    found = [f for f in sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)) + sorted(glob.glob(pattern))
             if "debug" not in f]
    assert len(found) == 1, f"{what}: нужен ровно один {pattern}, найдено {found}"
    return os.path.dirname(found[0])

ROLLOUTS = find_one("test_prompts.parquet", "роллауты")
SCORER = find_one("scorer_config.json", "scorer")
rollout_cfg = json.load(open(f"{ROLLOUTS}/config.json"))
scorer_cfg = json.load(open(f"{SCORER}/scorer_config.json"))
assert not rollout_cfg["debug"] and not scorer_cfg["train_meta"]["debug"], "подключены отладочные артефакты"
assert scorer_cfg["rollouts_cfg"] == rollout_cfg, "scorer обучен на других роллаутах"

GEN_KW = {k: v for k, v in rollout_cfg["gen_kw"].items() if k != "max_new_tokens"}
MAX_NEW_TOKENS = rollout_cfg["gen_kw"]["max_new_tokens"]

# Шарды полного прогона 03
evals = []
for f in sorted(glob.glob("/kaggle/input/**/run_meta.json", recursive=True)):
    m = json.load(open(f))
    if m.get("n_prompts") == 500 and m.get("rollouts_cfg") == rollout_cfg and glob.glob(f"{os.path.dirname(f)}/bon_*.parquet"):
        evals.append(os.path.dirname(f))
assert evals, "не найдены шарды полного прогона 03: подключи cd-hh-eval или Output ноутбука 03"
EVAL = evals[0]
EVAL_META = json.load(open(f"{EVAL}/run_meta.json"))
assert EVAL_META["scorer_train_meta"] == scorer_cfg["train_meta"], "шарды 03 посчитаны с другим scorer'ом"
BON_K_MAX, N_REF = EVAL_META["bon_k_max"], EVAL_META["n_ref"]
print("rollouts:", ROLLOUTS, "\nscorer:", SCORER, "\neval:", EVAL)

test = pd.read_parquet(f"{ROLLOUTS}/test_prompts.parquet").head(N_PROMPTS)
test["prompt_ids"] = test.prompt_ids.map(lambda x: x.tolist())
PIDS = np.sort(test.pid.values)
MSGS = test.set_index("pid").messages
n_shards = (len(test) + SHARD_SIZE - 1) // SHARD_SIZE

def load_eval(name):
    df = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{EVAL}/{name}_*.parquet"))], ignore_index=True)
    return df[df.pid.isin(PIDS)].sort_values(["pid", "sample"] if "sample" in df else "pid").reset_index(drop=True)

bon = load_eval("bon")
CD_NAMES = sorted({re.sub(r"_\d+\.parquet$", "", os.path.basename(f)) for f in glob.glob(f"{EVAL}/cd_M*_K*_*.parquet")},
                  key=lambda s: tuple(map(int, re.findall(r"\d+", s)))[::-1])
cds = {name: load_eval(name) for name in CD_NAMES}
assert bon.pid.nunique() == len(PIDS) and all((d.pid.values == PIDS).all() for d in cds.values()), "в шардах 03 не все промпты"
print(f"best-of-K: {len(bon)} ответов; CD: {', '.join(CD_NAMES)}")

# Защита от чужих шардов и подтягивание шардов прошлой сессии
FB_META = json.loads(json.dumps({"eval_meta": EVAL_META, "n_prompts": N_PROMPTS, "judges": JUDGES, "n_timing": N_TIMING,
                                 "temp_kw": TEMP_KW, "temp_cd": TEMP_CD, "temp_bon_k": TEMP_BON_K,
                                 "shard_size": SHARD_SIZE, "seed": SEED}))
meta_path = f"{OUT}/feedback_meta.json"
if os.path.exists(meta_path):
    assert json.load(open(meta_path)) == FB_META, f"В {OUT} результаты прогона с другими параметрами. Удали папку: !rm -rf {OUT}"
elif glob.glob(f"{OUT}/*.parquet"):
    raise RuntimeError(f"В {OUT} шарды неизвестного прогона. Удали папку: !rm -rf {OUT}")
else:
    json.dump(FB_META, open(meta_path, "w"), indent=2)
for prev in glob.glob("/kaggle/input/**/feedback_meta.json", recursive=True):
    if json.load(open(prev)) == FB_META:
        n = 0
        for f in glob.glob(os.path.join(os.path.dirname(prev), "*")):
            if f.endswith((".parquet", ".json")) and not os.path.exists(f"{OUT}/{os.path.basename(f)}"):
                shutil.copy(f, OUT); n += 1
        print(f"подтянул {n} файлов из {os.path.dirname(prev)}")

## Модели и замер времени

Функции те же, что в `03`. Дополнительно каждый вызов generate, scorer'а и RM может записывать время (с
`torch.cuda.synchronize`) и счётчики в `TIMER`, это нужно для раздела 4.

In [ ]:
tok = AutoTokenizer.from_pretrained(rollout_cfg["base_model"])
tok.padding_side = "left"
policy = AutoModelForCausalLM.from_pretrained(rollout_cfg["base_model"], dtype=DTYPE).to(GEN_DEVICE).eval()
eos = policy.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])

scorer = AutoModelForTokenClassification.from_pretrained(SCORER, dtype=DTYPE).to(AUX_DEVICE).eval()
scorer.config.pad_token_id = tok.pad_token_id

rm_tok = AutoTokenizer.from_pretrained(rollout_cfg["rm_model"])
rm_tok.padding_side = "right"
rm = AutoModelForSequenceClassification.from_pretrained(rollout_cfg["rm_model"], dtype=DTYPE, num_labels=1).to(AUX_DEVICE).eval()
rm.config.pad_token_id = rm_tok.pad_token_id

TIMER = None  # defaultdict(float) во время замеров

@contextmanager
def timed(key, **counts):
    if TIMER is None:
        yield; return
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)
    t0 = time.perf_counter()
    yield
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)
    TIMER[f"{key}_s"] += time.perf_counter() - t0
    TIMER[f"{key}_calls"] += 1
    for k, v in counts.items():
        TIMER[k] += v

def cut_at_eos(ids):
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            return ids[: i + 1], True
    return ids, False

@torch.no_grad()
def sample(seqs, n, max_new, gen_kw=GEN_KW):
    out = []
    per_call = max(1, GEN_SEQS // n)
    for b in range(0, len(seqs), per_call):
        batch = tok.pad({"input_ids": seqs[b:b + per_call]}, return_tensors="pt").to(GEN_DEVICE)
        with timed("gen", gen_seqs=batch["input_ids"].shape[0] * n):
            g = policy.generate(**batch, **gen_kw, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id)
        res = [cut_at_eos(o) for o in g[:, batch["input_ids"].shape[1]:].tolist()]
        if TIMER is not None:
            TIMER["gen_tokens"] += sum(len(r[0]) for r in res)
        out += res
    return out

@torch.no_grad()
def values(seqs):
    res = np.empty(len(seqs))
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in range(0, len(order), SCORER_BATCH):
        idx = order[k:k + SCORER_BATCH]
        L = max(len(seqs[i]) for i in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            ids[r, :len(seqs[i])] = torch.tensor(seqs[i]); att[r, :len(seqs[i])] = 1
        with timed("scorer", scorer_seqs=len(idx), scorer_tokens=int(att.sum())):
            v = scorer(input_ids=ids.to(AUX_DEVICE), attention_mask=att.to(AUX_DEVICE)).logits[..., 0].float()
            last = att.sum(1).to(AUX_DEVICE) - 1
            res[idx] = v[torch.arange(len(idx), device=AUX_DEVICE), last].cpu().numpy()
    return res

def chat_text(tk, messages, response):
    t = tk.apply_chat_template(json.loads(messages) + [{"role": "assistant", "content": response}], tokenize=False)
    return t[len(tk.bos_token):] if tk.bos_token and t.startswith(tk.bos_token) else t

@torch.no_grad()
def rewards(messages, responses, model=None, tk=None, bs=RM_BATCH, pbar=None):
    model, tk = (rm, rm_tok) if model is None else (model, tk)
    dev = next(model.parameters()).device
    texts = [chat_text(tk, m, r) for m, r in zip(messages, responses)]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    res = np.empty(len(texts))
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        enc = tk([texts[i] for i in idx], return_tensors="pt", padding=True).to(dev)
        with timed("rm", rm_seqs=len(idx), rm_tokens=int(enc["attention_mask"].sum())):
            res[idx] = model(**enc).logits[:, 0].float().cpu().numpy()
        if pbar is not None:
            pbar.update(len(idx))
    assert np.isfinite(res).all(), "inf/nan в reward: проблема с fp16"
    return res

def decode(ids):
    return tok.decode(ids, skip_special_tokens=True).strip()

def run_cd(prompt_ids, M, K, gen_kw=GEN_KW):
    n = len(prompt_ids)
    resp, fin, n_blocks, gen_tok = [[] for _ in range(n)], [False] * n, [0] * n, [0] * n
    done = [False] * n
    while not all(done):
        act = [i for i in range(n) if not done[i]]
        seqs = [prompt_ids[i] + resp[i] for i in act]
        max_new = min(M, max(MAX_NEW_TOKENS - len(resp[i]) for i in act))
        cands = sample(seqs, K, max_new, gen_kw)
        flat, meta = [], []
        for j, i in enumerate(act):
            rem = MAX_NEW_TOKENS - len(resp[i])
            for k in range(K):
                ids, f = cands[j * K + k]
                if len(ids) > rem:
                    ids, f = ids[:rem], False
                gen_tok[i] += len(ids)
                flat.append(seqs[j] + ids); meta.append((ids, f))
        v = values(flat)
        for j, i in enumerate(act):
            ids, f = meta[j * K + int(np.argmax(v[j * K:(j + 1) * K]))]
            resp[i] += ids; n_blocks[i] += 1; fin[i] = f
            done[i] = f or len(resp[i]) >= MAX_NEW_TOKENS
    return resp, fin, n_blocks, gen_tok

def shard_path(name, s):
    return f"{OUT}/{name}_{s:03d}.parquet"

def load_out(name):
    files = sorted(glob.glob(f"{OUT}/{name}_[0-9][0-9][0-9].parquet"))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else None

## Общие функции сравнения

Best-of-K считается как в `03`: 32 сэмпла режутся на 32/K непересекающихся групп, и результат усредняется по группам.
Сравнение CD с best-of-K идёт парным бутстрепом по промптам, как в разделе «Парные сравнения» в `03`.

In [ ]:
def ks_upto(kmax):
    return [2 ** i for i in range(int(math.log2(kmax)) + 1)]

def bon_curve(SEL, VAL, Ks):
    # (промпт, K): значение VAL у ответа с наибольшим SEL в группе, среднее по группам
    n = SEL.shape[0]
    out = []
    for K in Ks:
        pick = SEL.reshape(n, -1, K).argmax(2)
        out.append(np.take_along_axis(VAL.reshape(n, -1, K), pick[..., None], 2)[..., 0].mean(1))
    return np.stack(out, 1)

def win_vs(r, ref):
    # r: (n, m), ref: (n, n_ref) → (n, m): доля побед над опорными ответами π_ref, ничья 0.5
    return ((r[..., None] > ref[:, None]) + 0.5 * (r[..., None] == ref[:, None])).mean(-1)

def k_eq(c, curve, Ks):
    curve = np.maximum.accumulate(curve)
    return 2 ** np.interp(c, curve, np.log2(Ks))

def ci(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return f"[{lo:.2f}; {hi:.2f}]"

def compare(curve, c, Ks, bon_tok, cd_tok):
    # curve: (n, len(Ks)) best-of-K по промптам, c: (n,) CD по промптам
    n = len(c)
    idx = np.random.default_rng(0).integers(0, n, (N_BOOT, n))
    cb, bb = c[idx].mean(1), curve[idx].mean(1)
    ke = k_eq(c.mean(), curve.mean(0), Ks)
    j = int(np.searchsorted(bon_tok, cd_tok))
    over, j = j >= len(Ks), min(j, len(Ks) - 1)
    d = cb - bb[:, j]
    return {"K_eq": ("≥" if ke >= Ks[-1] else "") + f"{ke:.1f}", "K_eq_ci": ci([k_eq(x, y, Ks) for x, y in zip(cb, bb)]),
            "vs_tok": (">" if over else "") + f"best-of-{Ks[j]}", "Δ_tok": round(c.mean() - curve[:, j].mean(), 2),
            "Δ_tok_ci": ci(d), "P(Δ≤0)": round(float((d <= 0).mean()), 3)}

def cd_mk(name):
    return tuple(map(int, re.match(r"t?cd_M(\d+)_K(\d+)", name).groups()))

pd.set_option("display.width", 250)

## 1. Длина ответа и доля законченных (п. 4)

Для best-of-K берутся выбранные по RM ответы, для CD финальные ответы. «Законченный» значит, что ответ закончился EOS
до лимита в 256 токенов. Если RM систематически штрафует обрезанные ответы, а методы обрезаются с разной частотой,
часть разницы в reward объясняется этим, а не качеством. Reward только на законченных ответах проверяет это.
Оговорка: у best-of-K выбор сам по себе смещает долю законченных, потому что RM выбирает законченные ответы чаще.

In [ ]:
def length_row(method, df):
    return {"method": method, "resp_len": df.response_ids.map(len).mean(), "finished": df.finished.mean(),
            "reward": df.reward.mean(), "reward_finished": df[df.finished].reward.mean(),
            "reward_unfinished": df[~df.finished].reward.mean()}

pool = bon[bon["sample"] < BON_K_MAX]
rows = []
for K in ks_upto(BON_K_MAX):
    p = pool.assign(grp=pool["sample"] // K)
    rows.append(length_row("π_ref" if K == 1 else f"best-of-{K}", p.loc[p.groupby(["pid", "grp"]).reward.idxmax()]))
for name in CD_NAMES:
    M, K = cd_mk(name)
    rows.append(length_row(f"CD M={M} K={K}", cds[name]))
lengths = pd.DataFrame(rows)
lengths.to_csv(f"{OUT}/lengths.csv", index=False)
print(lengths.round(2).to_string(index=False))

## 2. Prefix accuracy без утечки полного ответа (п. 5)

В `02` для ответа короче t брался полный ответ, поэтому `pair_acc@t` при больших t частично измеряла точность на
полных ответах. Здесь пары, где хотя бы один ответ не длиннее t токенов, отбрасываются.

Две выборки пар:
- **val**: те же 435 пар, что в `02` (2 ответа на промпт, 500 промптов валидации);
- **test**: сэмплы best-of-K из `03` на test-промптах, которые scorer не видел. Пары образуют сэмплы (0,1), (2,3), …, (30,31),
  это до 16 пар на промпт, и погрешность получается намного меньше.

In [ ]:
@torch.no_grad()
def values_all(prompts, responses, desc):
    # V после 0..T токенов ответа
    seqs = [list(p) + list(r) for p, r in zip(prompts, responses)]
    out = [None] * len(seqs)
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in tqdm(range(0, len(order), SCORER_BATCH), desc=desc):
        idx = order[k:k + SCORER_BATCH]
        L = max(len(seqs[i]) for i in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            ids[r, :len(seqs[i])] = torch.tensor(seqs[i]); att[r, :len(seqs[i])] = 1
        v = scorer(input_ids=ids.to(AUX_DEVICE), attention_mask=att.to(AUX_DEVICE)).logits[..., 0].float().cpu().numpy()
        for r, i in enumerate(idx):
            out[i] = v[r, len(prompts[i]) - 1:len(seqs[i])]
    return out

def pair_acc_table(vals, lens, rew, pairs, label):
    rows = []
    for t in PAIR_TS + ["full"]:
        if t == "full":
            ok = [(vals[a][-1] > vals[b][-1]) == (rew[a] > rew[b]) for a, b in pairs]
            rows.append({"set": label, "t": t, "acc_old": np.mean(ok), "n_old": len(ok), "acc_strict": np.mean(ok), "n_strict": len(ok)})
            continue
        old = [(vals[a][min(t, lens[a])] > vals[b][min(t, lens[b])]) == (rew[a] > rew[b]) for a, b in pairs]
        strict = [(vals[a][t] > vals[b][t]) == (rew[a] > rew[b]) for a, b in pairs if lens[a] > t and lens[b] > t]
        rows.append({"set": label, "t": t, "acc_old": np.mean(old), "n_old": len(old),
                     "acc_strict": np.mean(strict) if strict else np.nan, "n_strict": len(strict)})
    df = pd.DataFrame(rows)
    df["ci_strict"] = 1.96 * np.sqrt(df.acc_strict * (1 - df.acc_strict) / df.n_strict)
    return df

f_pa = f"{OUT}/pair_acc.csv"
if RUN["pair_acc"] and not os.path.exists(f_pa):
    tr = pd.read_parquet(f"{ROLLOUTS}/train.parquet")
    vp = np.random.default_rng(scorer_cfg["train_meta"]["seed"]).permutation(tr.pid.unique())
    val = tr[tr.pid.isin(set(vp[: int(len(vp) * 0.05)].tolist()))].reset_index(drop=True)
    assert len(val) == scorer_cfg["train_meta"]["n_val"]
    del tr
    vv = values_all(val.prompt_ids.tolist(), val.response_ids.tolist(), "val")
    vpairs = [(g.index[0], g.index[1]) for _, g in val.groupby("pid")
              if len(g) == 2 and abs(g.reward.iloc[0] - g.reward.iloc[1]) > PAIR_MIN_GAP]
    t1 = pair_acc_table(vv, val.response_ids.map(len).values, val.reward.values, vpairs, "val")

    tp = pool.merge(test[["pid", "prompt_ids"]], on="pid").sort_values(["pid", "sample"]).reset_index(drop=True)
    tv = values_all(tp.prompt_ids.tolist(), tp.response_ids.tolist(), "test")
    tpairs = [(a, a + 1) for a in range(0, len(tp), 2) if abs(tp.reward[a] - tp.reward[a + 1]) > PAIR_MIN_GAP]
    t2 = pair_acc_table(tv, tp.response_ids.map(len).values, tp.reward.values, tpairs, "test")
    pd.concat([t1, t2]).to_csv(f_pa, index=False)
if os.path.exists(f_pa):
    print(pd.read_csv(f_pa).round(3).to_string(index=False))

## 3. Другие судьи (п. 3)

Все ответы из `03` переоцениваются независимыми RM. Это 36 сэмплов π_ref на промпт и ответы всех 9 конфигураций CD.
Best-of-K **выбирает** ответ по-прежнему по Skywork 0.6B (так метод работает на инференсе), а **оценивается** выбранный
ответ судьёй. Если выигрыш CD — подгонка под Skywork 0.6B, у независимого судьи он уменьшится или исчезнет. Та же
подгонка видна и у best-of-K: его кривая у другого судьи растёт медленнее.

Судьи считаются параллельно, по одному на каждой GPU. Каждый шард сохраняется сразу.

In [ ]:
SRC_ALL = ["bon"] + CD_NAMES

def judge_df(src):
    return bon if src == "bon" else cds[src]

def judge_todo(jn):
    return [(src, sh) for src in SRC_ALL for sh in range(n_shards) if not os.path.exists(shard_path(f"judge_{jn}_{src}", sh))]

def load_judge(jpath, device):
    jtok = AutoTokenizer.from_pretrained(jpath)
    jtok.padding_side = "right"
    # .to(dtype=...) на всякий случай: часть весов не должна остаться в fp32
    jm = AutoModelForSequenceClassification.from_pretrained(jpath, dtype=DTYPE, num_labels=1).to(device=device, dtype=DTYPE).eval()
    jm.config.pad_token_id = jtok.pad_token_id
    return jm, jtok

def run_judge(jn, jm, jtok, position):
    todo = judge_todo(jn)
    shard_pids = [set(test.pid.iloc[sh * SHARD_SIZE:(sh + 1) * SHARD_SIZE]) for sh in range(n_shards)]
    parts = {(src, sh): judge_df(src)[judge_df(src).pid.isin(shard_pids[sh])] for src, sh in todo}
    pbar = tqdm(total=sum(len(d) for d in parts.values()), desc=jn, unit="resp", position=position)
    for (src, sh), d in parts.items():
        r = rewards(MSGS.loc[d.pid.values].tolist(), d.response.tolist(), jm, jtok, JUDGE_BATCH, pbar=pbar)
        out = pd.DataFrame({"pid": d.pid.values, "sample": d["sample"].values if "sample" in d else 0, "reward": r})
        out.to_parquet(shard_path(f"judge_{jn}_{src}", sh))
    pbar.close()

if RUN["judges"]:
    # Модели грузим по очереди в основном потоке: from_pretrained глобально меняет default dtype torch,
    # и параллельная загрузка в потоках оставляет часть весов в fp32. Параллельно, по судье на GPU, идёт только подсчёт.
    from concurrent.futures import ThreadPoolExecutor
    devs = [GEN_DEVICE, AUX_DEVICE]
    todo_j = [jn for jn in JUDGES if judge_todo(jn)]
    for i in range(0, len(todo_j), len(devs)):
        group = todo_j[i:i + len(devs)]
        loaded = {jn: load_judge(JUDGES[jn], devs[k]) for k, jn in enumerate(group)}
        with ThreadPoolExecutor(len(group)) as ex:
            futs = [ex.submit(run_judge, jn, *loaded[jn], k) for k, jn in enumerate(group)]
            for f in futs:
                f.result()
        del loaded
        torch.cuda.empty_cache()

In [ ]:
BASE = "skywork_0.6b"

def judge_scores(jn, src):
    if jn == BASE:
        d = judge_df(src)
        return d.assign(sample=d["sample"] if "sample" in d else 0)[["pid", "sample", "reward"]]
    d = load_out(f"judge_{jn}_{src}")
    return None if d is None else d.sort_values(["pid", "sample"]).reset_index(drop=True)

n = len(PIDS)
Ks = ks_upto(BON_K_MAX)
SEL = bon.reward.values.reshape(n, -1)[:, :BON_K_MAX]  # выбор best-of-K всегда по Skywork 0.6B
bon_tok = np.array(Ks) * pool.response_ids.map(len).mean()
jrows, corrs = [], {}
for jn in [BASE] + list(JUDGES):
    jb = judge_scores(jn, "bon")
    if jb is None or len(jb) != len(bon) or any(judge_scores(jn, s) is None for s in CD_NAMES):
        print(f"{jn}: не все шарды посчитаны, пропускаю"); continue
    A = jb.reward.values.reshape(n, -1)
    JR, REF = A[:, :BON_K_MAX], A[:, BON_K_MAX:]
    corrs[jn] = pd.Series(jb.reward.values).rank().corr(pd.Series(bon.reward.values).rank())  # Spearman
    cr, cw = bon_curve(SEL, JR, Ks), bon_curve(SEL, win_vs(JR, REF), Ks)
    for j, K in enumerate(Ks):
        jrows.append({"judge": jn, "method": "π_ref" if K == 1 else f"best-of-{K}", "reward": cr[:, j].mean(), "win": cw[:, j].mean()})
    for src in CD_NAMES:
        M, K = cd_mk(src)
        c = judge_scores(jn, src).reward.values
        w = win_vs(c[:, None], REF)[:, 0]
        jrows.append({"judge": jn, "method": f"CD M={M} K={K}", "reward": c.mean(), "win": w.mean(),
                      **compare(cr, c, Ks, bon_tok, cds[src].gen_tokens.mean())})

if jrows:
    judges = pd.DataFrame(jrows)
    judges.to_csv(f"{OUT}/judges.csv", index=False)
    print("Spearman с Skywork 0.6B на всех сэмплах π_ref:", {k: round(v, 3) for k, v in corrs.items()})
    print("\nWin rate против π_ref (судья сравнивает ответ метода с 4 независимыми сэмплами π_ref):")
    print(judges.pivot(index="method", columns="judge", values="win").reindex(judges.method.unique()).round(3).to_string())
    cdj = judges[judges.method.str.startswith("CD")]
    print("\nCD против best-of-K по оценке каждого судьи (K_eq и Δ при равных токенах в единицах reward судьи):")
    print(cdj[["judge", "method", "reward", "win", "K_eq", "K_eq_ci", "vs_tok", "Δ_tok", "Δ_tok_ci", "P(Δ≤0)"]].round(3).to_string(index=False))

## 4. Реальная стоимость (п. 2)

На первых 50 test-промптах каждый метод запускается заново, с теми же батчами, что в `03`: 50 промптов за раз,
до 128 последовательностей в одном generate. Время меряется с `torch.cuda.synchronize`. Учитывается всё, что метод
делает на инференсе:
- best-of-K: генерация K ответов и оценка всех K ответов RM (Skywork 0.6B) для выбора;
- CD: генерация блоков и оценка K кандидатов scorer'ом (0.5B) на каждом блоке. RM в CD не нужна.

Важно: scorer не использует KV-кэш, на каждом блоке он заново читает промпт и весь префикс. Колонка `scorer_tok/prompt`
показывает, сколько это токенов.

In [ ]:
if RUN["timing"]:
    tp_ = test.head(N_TIMING)
    P_, MS_ = tp_.prompt_ids.tolist(), tp_.messages.tolist()
    TIMER = None
    sample(P_[:2], 2, 16); values([P_[0]]); rewards(MS_[:1], ["Hi"])  # прогрев CUDA
    methods = [("bon", K, None) for K in TIMING_BON_KS] + [("cd", K, M) for M, K in sorted(cd_mk(s) for s in CD_NAMES)]
    for kind, K, M in tqdm(methods, desc="timing"):
        f = f"{OUT}/timing_{kind}_K{K}" + (f"_M{M}" if M else "") + ".json"
        if os.path.exists(f):
            continue
        torch.manual_seed(SEED * 100_003 + 7 * K + (M or 0))
        TIMER = defaultdict(float)
        t0 = time.perf_counter()
        if kind == "bon":
            res = sample(P_, K, MAX_NEW_TOKENS)
            r = rewards([m for m in MS_ for _ in range(K)], [decode(ids) for ids, _ in res])
            _ = r.reshape(-1, K).argmax(1)
        else:
            run_cd(P_, M, K)
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)
        TIMER["total_s"] = time.perf_counter() - t0
        json.dump(dict(TIMER) | {"method": "best-of-K" if kind == "bon" else "CD", "K": K, "M": M, "n_prompts": len(P_)}, open(f, "w"))
        TIMER = None

tf = sorted(glob.glob(f"{OUT}/timing_*.json"))
if tf:
    t = pd.DataFrame([json.load(open(f)) for f in tf])
    for c in ["gen_s", "scorer_s", "rm_s", "scorer_calls", "rm_calls", "scorer_tokens", "rm_tokens", "scorer_seqs", "rm_seqs"]:
        t[c] = t.get(c, 0)
    t = t.fillna(0)
    np_ = t.n_prompts
    timing = pd.DataFrame({
        "method": t.method, "M": t.M.astype(int), "K": t.K.astype(int),
        "total s/prompt": t.total_s / np_,
        "gen s/prompt": t.gen_s / np_, "scorer s/prompt": t.scorer_s / np_, "rm s/prompt": t.rm_s / np_,
        "other s/prompt": (t.total_s - t.gen_s - t.scorer_s - t.rm_s) / np_,
        "gen_tok/prompt": t.gen_tokens / np_, "generate calls": t.gen_calls.astype(int),
        "s/generate call": t.gen_s / t.gen_calls,
        "scorer seqs/prompt": t.scorer_seqs / np_, "scorer_tok/prompt": t.scorer_tokens / np_,
        "s/scorer batch": (t.scorer_s / t.scorer_calls.replace(0, np.nan)),
    }).sort_values(["method", "M", "K"])
    timing.to_csv(f"{OUT}/timing.csv", index=False)
    print(timing.round(3).to_string(index=False))

## 5. Температура T=0.7, top-p=0.9 (п. 6)

Те же 500 test-промптов. Best-of-K (32 сэмпла) и 3 конфигурации CD сэмплируются из π_ref с T=0.7 и top-p=0.9.
Scorer тот же, обученный на ответах при T=1. Reward считается Skywork 0.6B, как в `03`.
Оценка KL здесь не приводится: она относится к π_ref с T=0.7, а не к исходной π_ref, поэтому её нельзя сравнивать с `03`.

In [ ]:
TEMP_KW_FULL = GEN_KW | TEMP_KW
print("генерация:", TEMP_KW_FULL)
if RUN["temperature"]:
    pbar = tqdm(total=len(test), desc="T=0.7 best-of-K", unit="prompt")
    for s in range(n_shards):
        sh = test.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
        if os.path.exists(shard_path("tbon", s)):
            pbar.update(len(sh)); continue
        torch.manual_seed(SEED * 100_003 + 50_000 + s)
        res = sample(sh.prompt_ids.tolist(), TEMP_BON_K, MAX_NEW_TOKENS, TEMP_KW_FULL)
        rows = []
        for j, p in enumerate(sh.itertuples()):
            for k in range(TEMP_BON_K):
                ids, fin = res[j * TEMP_BON_K + k]
                rows.append({"pid": p.pid, "sample": k, "messages": p.messages, "seq": p.prompt_ids + ids,
                             "response_ids": ids, "finished": fin, "response": decode(ids)})
        df = pd.DataFrame(rows)
        df["reward"] = rewards(df.messages, df.response)
        df["v"] = values(df.seq.tolist())
        df.drop(columns=["seq", "messages"]).to_parquet(shard_path("tbon", s))
        pbar.update(len(sh))
    pbar.close()

    for ci_, (M, K) in enumerate(TEMP_CD):
        name = f"tcd_M{M}_K{K}"
        pbar = tqdm(total=len(test), desc=f"T=0.7 {name}", unit="prompt")
        for s in range(n_shards):
            sh = test.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
            if os.path.exists(shard_path(name, s)):
                pbar.update(len(sh)); continue
            torch.manual_seed(SEED * 100_003 + 60_000 + 1000 * ci_ + s)
            resp, fin, nb, gt = run_cd(sh.prompt_ids.tolist(), M, K, TEMP_KW_FULL)
            df = pd.DataFrame({"pid": sh.pid.values, "response_ids": resp, "finished": fin, "n_blocks": nb, "gen_tokens": gt,
                               "response": [decode(r) for r in resp]})
            df["reward"] = rewards(sh.messages.tolist(), df.response.tolist())
            df.to_parquet(shard_path(name, s))
            pbar.update(len(sh))
        pbar.close()

In [ ]:
tb = load_out("tbon")
if tb is not None and tb.pid.nunique() == n:
    tb = tb.sort_values(["pid", "sample"]).reset_index(drop=True)
    Kt = ks_upto(TEMP_BON_K)
    sets = {"T=1": pool[pool["sample"] < TEMP_BON_K], "T=0.7": tb}
    curves, toks, rows = {}, {}, []
    for lab, d in sets.items():
        R = d.reward.values.reshape(n, TEMP_BON_K)
        curves[lab] = bon_curve(R, R, Kt)
        toks[lab] = np.array(Kt) * d.response_ids.map(len).mean()
        for K in Kt:
            p = d.assign(grp=d["sample"] // K)
            rows.append({"T": lab, **length_row("π_ref" if K == 1 else f"best-of-{K}", p.loc[p.groupby(["pid", "grp"]).reward.idxmax()]),
                         "gen_tok": K * d.response_ids.map(len).mean()})
    for M, K in TEMP_CD:
        for lab, d in [("T=1", cds.get(f"cd_M{M}_K{K}")), ("T=0.7", load_out(f"tcd_M{M}_K{K}"))]:
            if d is None or d.pid.nunique() != n:
                continue
            d = d.sort_values("pid")
            rows.append({"T": lab, **length_row(f"CD M={M} K={K}", d), "gen_tok": d.gen_tokens.mean(),
                         **compare(curves[lab], d.reward.values, Kt, toks[lab], d.gen_tokens.mean())})
    temp = pd.DataFrame(rows)
    temp.to_csv(f"{OUT}/temperature.csv", index=False)
    print("K_eq и Δ считаются относительно best-of-K при той же температуре\n")
    cols = ["T", "method", "reward", "gen_tok", "resp_len", "finished", "reward_finished"] + \
           [c for c in ["K_eq", "K_eq_ci", "vs_tok", "Δ_tok", "Δ_tok_ci", "P(Δ≤0)"] if c in temp]
    print(temp[cols].round(2).to_string(index=False))

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for lab, col in [("T=1", "black"), ("T=0.7", "tab:red")]:
        ax.plot(toks[lab], curves[lab].mean(0), "o-", color=col, label=f"best-of-K, {lab}")
        d = temp[(temp["T"] == lab) & temp.method.str.startswith("CD")]
        ax.plot(d.gen_tok, d.reward, "^", color=col, ms=9, label=f"CD, {lab}")
        for _, r in d.iterrows():
            ax.annotate(r.method[3:], (r.gen_tok, r.reward), fontsize=7, xytext=(4, -10), textcoords="offset points")
    ax.set_xscale("log", base=2); ax.set_xlabel("сгенерировано токенов на промпт"); ax.set_ylabel("reward (Skywork 0.6B)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/temperature.png", dpi=150); plt.show()

## Сохранение результатов в Kaggle Dataset

Загружает всё из `cd_feedback` (шарды, csv, png) в приватный датасет `cd-hh-feedback`. Нужен секрет `KAGGLE_API_TOKEN`.

In [ ]:
!pip install -q -U kaggle

import subprocess
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")

USERNAME = "karimvafin"
SLUG = "cd-hh-feedback-debug" if DEBUG else "cd-hh-feedback"

def kaggle_cli(*args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def upload_dir(src, slug, extra_files=(), message="update"):
    # Kaggle CLI без --dir-mode пропускает подпапки, поэтому всё кладём в один плоский каталог
    stage = f"{WORK}/upload_{slug}"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(src, stage)
    for f in extra_files:
        shutil.copy(f, stage)
    ds_id = f"{USERNAME}/{slug}"
    json.dump({"title": slug, "id": ds_id, "licenses": [{"name": "CC0-1.0"}]}, open(f"{stage}/dataset-metadata.json", "w"))
    exists = kaggle_cli("datasets", "status", ds_id)[0] == 0
    code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message) if exists else kaggle_cli("datasets", "create", "-p", stage)
    if code != 0 and not exists:
        # status мог ошибочно сказать, что датасета нет (например, он остался от прошлой попытки): пробуем как новую версию
        print(out[-2000:], "\n→ create не удался, пробую version")
        code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message)
        exists = True
    print(out[-2000:])
    assert code == 0, "загрузка не удалась, см. вывод выше; файлы остались в " + stage
    shutil.rmtree(stage, ignore_errors=True)
    print(f"{'новая версия' if exists else 'создан'}: https://www.kaggle.com/datasets/{ds_id}")

upload_dir(OUT, SLUG, message="feedback 09.11")